In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

all_sheets = pd.read_excel('/content/drive/MyDrive/Cadetx /ev_charging_dataset.xlsx', sheet_name=None)
sessions_df = all_sheets['sessions']
stations_df = all_sheets['stations']

print(sessions_df.shape)

(294024, 15)


In [3]:
station_performance = sessions_df.groupby('station_id').agg(
    total_sessions=('session_id', 'count'),
    total_revenue=('total_cost', 'sum'),
    unique_customers=('customer_id', 'nunique'),
    total_kwh=('energy_kwh', 'sum')
).reset_index()

print(station_performance.head())
print(station_performance.shape)

  station_id  total_sessions  total_revenue  unique_customers  total_kwh
0    STN_001           15162      359914.16              4585   620951.6
1    STN_002           15276      545458.72              4637   940999.0
2    STN_003           15321      424603.28              4659   733802.6
3    STN_004           15121      417831.11              4655   720723.5
4    STN_005           15160      406249.04              4621   702048.5
(33, 5)


In [4]:
station_performance['sessions_per_customer'] = station_performance['total_sessions'] / station_performance['unique_customers']
print(station_performance.head())

  station_id  total_sessions  total_revenue  unique_customers  total_kwh  \
0    STN_001           15162      359914.16              4585   620951.6   
1    STN_002           15276      545458.72              4637   940999.0   
2    STN_003           15321      424603.28              4659   733802.6   
3    STN_004           15121      417831.11              4655   720723.5   
4    STN_005           15160      406249.04              4621   702048.5   

   sessions_per_customer  
0               3.306870  
1               3.294371  
2               3.288474  
3               3.248335  
4               3.280675  


In [5]:
# Normalize each metric to a 0-100 scale
for col in ['total_sessions', 'total_revenue', 'sessions_per_customer', 'total_kwh']:
    min_val = station_performance[col].min()
    max_val = station_performance[col].max()
    station_performance[col + '_score'] = ((station_performance[col] - min_val) / (max_val - min_val)) * 100

print(station_performance[['station_id', 'total_sessions_score', 'total_revenue_score', 'sessions_per_customer_score', 'total_kwh_score']].head())

  station_id  total_sessions_score  total_revenue_score  \
0    STN_001             98.470271            58.489517   
1    STN_002             99.567058           100.000000   
2    STN_003            100.000000            72.961925   
3    STN_004             98.075813            71.446839   
4    STN_005             98.451029            68.855670   

   sessions_per_customer_score  total_kwh_score  
0                   100.000000        58.625056  
1                    99.121493       100.000000  
2                    98.706981        73.214157  
3                    95.885749        71.523323  
4                    98.158831        69.109064  


In [7]:
station_performance['overall_score'] = station_performance[
    ['total_sessions_score', 'total_revenue_score', 'sessions_per_customer_score', 'total_kwh_score']
].mean(axis=1)

print(station_performance[['station_id', 'overall_score']].sort_values('overall_score', ascending=False))

   station_id  overall_score
1     STN_002      99.672138
7     STN_008      93.504181
5     STN_006      87.155605
2     STN_003      86.220766
3     STN_004      84.232931
4     STN_005      83.643649
6     STN_007      81.253459
0     STN_001      78.896211
11    STN_012      54.881401
16    STN_017      49.323272
13    STN_014      48.208042
9     STN_010      46.860428
10    STN_011      46.692659
12    STN_013      46.043167
14    STN_015      43.446443
15    STN_016      43.443059
8     STN_009      38.850600
27    STN_028      10.863707
18    STN_019       9.134268
28    STN_029       8.092183
32    STN_033       7.671730
26    STN_027       6.642892
17    STN_018       6.016756
24    STN_025       5.656586
31    STN_032       4.701854
19    STN_020       4.225654
25    STN_026       4.199195
23    STN_024       4.067286
22    STN_023       4.004101
30    STN_031       3.878229
29    STN_030       1.826100
20    STN_021       1.538465
21    STN_022       1.469434


In [8]:
low_score_stations = station_performance[station_performance['overall_score'] < 15]['station_id'].tolist()
print(stations_df[stations_df['station_id'].isin(low_score_stations)][['station_id', 'activation_year']])

   station_id  activation_year
17    STN_018             2024
18    STN_019             2024
19    STN_020             2024
20    STN_021             2024
21    STN_022             2024
22    STN_023             2024
23    STN_024             2024
24    STN_025             2024
25    STN_026             2024
26    STN_027             2024
27    STN_028             2024
28    STN_029             2024
29    STN_030             2024
30    STN_031             2024
31    STN_032             2024
32    STN_033             2024


In [9]:
station_performance.to_csv('/content/drive/MyDrive/Cadetx /station_performance_ranking.csv', index=False)
print("Saved!")

Saved!
